# Glacier du Mont Miné — visualize the results

*Session 4. The step-by-step guide is in the slides `session4/lecture-4-mont-mine.md`.*

This notebook **only reads** the run folders `outputs/<run>/`: run IGM in the **terminal**, from this folder (`cd ~/work/mont-mine`: your copy, **outside** the course repository, made with `cp -r ~/work/glamod/session4/mont-mine ~/work/mont-mine`), then run the cells below. Re-run them as often as you like.

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/work/glamod/utils"))   # course plotting helpers (utils/helper.py)
sys.path.insert(0, os.path.abspath("../../utils"))               # (same, when run inside the course folder)
from helper import *

RGI_ID = "RGI2000-v7.0-G-11-00907"   # Glacier du Mont Miné
INPUT = f"data/input_{RGI_ID}.nc"
SAMPLES = "Samples_2024.csv"          # sample sites (ID, CH1903 x/y, UTM32 x/y): indicative, replace by YOUR file

## 1. Ice-flow calibration (as in Session 2)

Grid search of ice softness (`arrhenius`) × basal sliding (`tau_ref`): 1-year runs from **today's** glacier, one per couple (3 × 4 = 12 runs; the first one downloads the data).

▶ **In the terminal**:

```bash
igm_run -m +experiment=params_sensitivity processes.iceflow.physics.viscosity.arrhenius=25,50,100 processes.iceflow.physics.sliding.tau_ref=0.1,0.2,0.4,0.8
```

Each map shows the speed difference (modelled − observed, Millan et al., 2022) of one run: pick the couple with a **bias close to 0** and the smallest RMSE (refine around it if needed: re-run with other values, the folder is overwritten). `experiment/params.yaml` uses the best couple of this grid (`arrhenius` = 100, `tau_ref` = 0.4): if you find a better one by refining, put it in its `iceflow → physics` block.

In [ ]:
plot_input_data(INPUT)
runs, labels = sweep_runs("outputs/params_sensitivity")   # the 12 runs
compare_velocity_misfit(INPUT, runs, year=2024)

> ✅ **Ice flow** (double-click to edit): `arrhenius` = 100  `tau_ref` = 0.4 (best of the 3 × 4 grid; refined: ...)

## 2. The reference run

Terminal (with the ice flow of section 1 in `params.yaml`): `igm_run +experiment=params` ➡️ results in `outputs/params/` (the run folder has the name of the parameter file)

The SMB profile (ELA 2900 m), then the run:

In [ ]:
plot_smb_simple(ela=2900, gradabl=0.009, gradacc=0.005, accmax=2.0, zmin=1800, zmax=3800)

In [ ]:
run = "outputs/params"          # <- the run you want to look at
show_dashboard(run, n=4)
analyze_time_series(os.path.join(run, "output_ts.nc"))

## 3. Ice cover at the sample sites

The samples provided (`Samples_2024.csv`) are **only indicative**, to get things running: put **your own samples** in a file of the same format (ID, CH1903 x/y, UTM32 x/y) in this folder, and set `SAMPLES` in the first cell.

In [ ]:
plot_samples_map(INPUT, SAMPLES)                   # today's glacier and the sample sites
plot_samples_map(os.path.join(run, "output.nc"), SAMPLES, year=1850)   # modelled LIA glacier

In [ ]:
animate_glacier_evolution(os.path.join(run, "output.nc"), variable="thk")

## 4. Modelled deglaciation years vs measured ages

Ice thickness through time at each site, and the modelled deglaciation year (last time the site is covered by ice). Enter **your** measured ages in `ages` (calendar years, or years before 2024).

In [ ]:
ages = {}   # e.g. {"24FER1": 1890, "24FER2": 1925, ...}  <- measured ages
table = plot_thk_at_samples(os.path.join(run, "output.nc"), SAMPLES, ages=ages)
table

### Calibration of the ELA history

Terminal: copy `experiment/params.yaml` to `experiment/my.params.yaml`, change **only the ELA history** (`ela` column of the `smb` array), then `igm_run +experiment=my.params` ➡️ `outputs/my.params/`. Change one thing at a time until the modelled deglaciation years match your sample ages.

In [ ]:
compare_runs(["outputs/params", "outputs/my.params"], ["reference", "my ELA history"])
plot_thk_at_samples("outputs/my.params/output.nc", SAMPLES, ages=ages)

## 5. Second set of sample sites (indicative)

In [ ]:
plot_thk_at_samples(os.path.join(run, "output.nc"), "Samples_XXXX.csv", ages=ages)